# RetailPulse — Clean Data Engineering Pipeline

**Layers:** Bronze → Silver → Gold

This notebook is the cleaned, reproducible version of the original work. Exploratory/repeated cells were removed, Gold transformations are consolidated, SCD Type 2 history is preserved, and validation is performed before the Gold tables are published.

## 1. Spark & Configuration

One Spark session and one configuration block are used throughout the notebook.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("RetailPulse-ETL")
    .getOrCreate()
)

BRONZE_DB = "retailpulse_bronze"
SILVER_DB = "retailpulse_silver"
GOLD_DB = "retailpulse_gold"

BRONZE_BASE_PATH = "hdfs://namenode:8020/data/retailpulse/bronze"
SILVER_BASE_PATH = "hdfs://namenode:8020/data/retailpulse/silver"
GOLD_BASE_PATH = "hdfs://namenode:8020/data/retailpulse/gold"

print("Spark version:", spark.version)
print("Bronze:", BRONZE_BASE_PATH)
print("Silver:", SILVER_BASE_PATH)
print("Gold:", GOLD_BASE_PATH)

Spark version: 2.4.1
Bronze: hdfs://namenode:8020/data/retailpulse/bronze
Silver: hdfs://namenode:8020/data/retailpulse/silver
Gold: hdfs://namenode:8020/data/retailpulse/gold


## 2. Load Bronze Tables

The Bronze layer is treated as the raw landing layer. No exploratory writes or repeated streaming queries are performed here.

In [2]:
bronze_tables = [
    "customers",
    "fulfillment_events",
    "inventory_snapshots",
    "order_items",
    "orders",
    "payments",
    "products",
    "stores",
    "streaming_events"
]

def read_bronze_tables(tables):
    result = {}
    for table in tables:
        result[table] = spark.table(f"{BRONZE_DB}.{table}")
        print(f"Loaded: {BRONZE_DB}.{table}")
    return result

dfs = read_bronze_tables(bronze_tables)

Loaded: retailpulse_bronze.customers
Loaded: retailpulse_bronze.fulfillment_events
Loaded: retailpulse_bronze.inventory_snapshots
Loaded: retailpulse_bronze.order_items
Loaded: retailpulse_bronze.orders
Loaded: retailpulse_bronze.payments
Loaded: retailpulse_bronze.products
Loaded: retailpulse_bronze.stores
Loaded: retailpulse_bronze.streaming_events


## 3. Bronze Data Quality Checks

These checks identify null primary keys and duplicate primary keys before transformation.

In [3]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "stores": "store_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "fulfillment_events": "fulfillment_event_id",
    "inventory_snapshots": "inventory_snapshot_id"
}

def validate_primary_keys(dataframes, keys):
    for table, pk in keys.items():
        df = dataframes[table]
        nulls = df.filter(F.col(pk).isNull()).count()
        duplicates = (
            df.groupBy(pk).count()
              .filter(F.col(pk).isNotNull() & (F.col("count") > 1))
              .count()
        )
        print(f"{table:25} | null {pk}: {nulls:,} | duplicate {pk}: {duplicates:,}")

validate_primary_keys(dfs, primary_keys)

customers                 | null customer_id: 0 | duplicate customer_id: 0
products                  | null product_id: 0 | duplicate product_id: 0
stores                    | null store_id: 0 | duplicate store_id: 0
orders                    | null order_id: 0 | duplicate order_id: 0
order_items               | null order_item_id: 0 | duplicate order_item_id: 0
payments                  | null payment_id: 0 | duplicate payment_id: 0
fulfillment_events        | null fulfillment_event_id: 0 | duplicate fulfillment_event_id: 0
inventory_snapshots       | null inventory_snapshot_id: 0 | duplicate inventory_snapshot_id: 0


## 4.2 Customers — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [4]:
from pyspark.sql import functions as F


# ============================================================
# CUSTOMERS - DEEP CLEANING
# Bronze → Silver
# ============================================================

customers_df = dfs["customers"]


# ------------------------------------------------------------
# 1. Select required columns
# ------------------------------------------------------------

customers_silver_df = customers_df.select(
    "customer_id",
    "full_name",
    "email",
    "phone",
    "country_code",
    "signup_at",
    "updated_at"
)


# ------------------------------------------------------------
# 2. Clean string columns
#    - trim spaces
#    - "null" → NULL
#    - empty string → NULL
# ------------------------------------------------------------

string_columns = [
    "full_name",
    "email",
    "phone",
    "country_code"
]

for column in string_columns:

    customers_silver_df = customers_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column).cast("string"))) == "null") |
            (F.trim(F.col(column).cast("string")) == ""),
            None
        ).otherwise(
            F.trim(F.col(column).cast("string"))
        )
    )


# ------------------------------------------------------------
# 3. Standardize full_name
#    Ahmed ALI → Ahmed Ali
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "full_name",
    F.initcap(F.lower(F.col("full_name")))
)


# ------------------------------------------------------------
# 4. Standardize email
#    Ahmed@GMAIL.COM → ahmed@gmail.com
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "email",
    F.lower(F.trim(F.col("email")))
)


# ------------------------------------------------------------
# 5. Standardize phone
#    Remove:
#    spaces
#    -
#    (
#    )
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "phone",
    F.regexp_replace(
        F.col("phone"),
        r"[\s\-\(\)]",
        ""
    )
)


# ------------------------------------------------------------
# 6. Standardize country_code
#
#    Egypt / egypt / EGY / eg → EG
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "country_code",
    F.lower(F.trim(F.col("country_code")))
)

customers_silver_df = customers_silver_df.withColumn(
    "country_code",
    F.when(
        F.col("country_code").isin(
            "egypt",
            "egy",
            "eg"
        ),
        "EG"
    ).otherwise(
        F.upper(F.col("country_code"))
    )
)


# ------------------------------------------------------------
# 7. Cast customer_id
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "customer_id",
    F.col("customer_id").cast("integer")
)


# ------------------------------------------------------------
# 8. Convert timestamps
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "signup_at",
    F.to_timestamp("signup_at")
)

customers_silver_df = customers_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# 9. Remove invalid email values
#
#    Any email that doesn't match the expected format
#    becomes NULL
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        F.col("email")
    ).otherwise(None)
)


# ------------------------------------------------------------
# 10. Remove records with NULL values
#
#     After all cleaning operations,
#     drop rows containing ANY NULL.
# ------------------------------------------------------------

before_drop = customers_silver_df.count()

customers_silver_df = customers_silver_df.dropna(
    how="any"
)

after_drop = customers_silver_df.count()


# ------------------------------------------------------------
# 11. Remove duplicate customers
#
#     customer_id is the Primary Key.
#     Keep one record per customer_id.
# ------------------------------------------------------------

before_duplicates = customers_silver_df.count()

customers_silver_df = customers_silver_df.dropDuplicates(
    ["customer_id"]
)

after_duplicates = customers_silver_df.count()


# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

print("=" * 70)
print("CUSTOMERS DEEP CLEANING RESULT")
print("=" * 70)

print(f"Rows before NULL removal      : {before_drop}")
print(f"Rows after NULL removal       : {after_drop}")
print(f"NULL rows removed             : {before_drop - after_drop}")

print()

print(f"Rows before duplicate removal : {before_duplicates}")
print(f"Rows after duplicate removal  : {after_duplicates}")
print(f"Duplicate rows removed        : {before_duplicates - after_duplicates}")

print()

print("Final Schema:")
customers_silver_df.printSchema()

print("Final Data:")
customers_silver_df.show(
    10,
    truncate=False
)


# ------------------------------------------------------------
# 13. Final NULL check
# ------------------------------------------------------------

print("Final NULL Check:")

customers_silver_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in customers_silver_df.columns
]).show()

CUSTOMERS DEEP CLEANING RESULT
Rows before NULL removal      : 40000
Rows after NULL removal       : 39415
NULL rows removed             : 585

Rows before duplicate removal : 39415
Rows after duplicate removal  : 39415
Duplicate rows removed        : 0

Final Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- signup_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

Final Data:
+-----------+-------------+------------------------+------------+------------+-------------------+-------------------+
|customer_id|full_name    |email                   |phone       |country_code|signup_at          |updated_at         |
+-----------+-------------+------------------------+------------+------------+-------------------+-------------------+
|148        |Customer 148 |customer148@example.com |+20100000148

## 4.4 Products — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [5]:
from pyspark.sql import functions as F


# ============================================================
# PRODUCTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

products_df = dfs["products"]


# ------------------------------------------------------------
# 1. Select required columns
# ------------------------------------------------------------

products_silver_df = products_df.select(
    "product_id",
    "sku",
    "product_name",
    "category",
    "unit_cost",
    "list_price",
    "updated_at"
)


# ------------------------------------------------------------
# 2. Clean string columns
#    - trim spaces
#    - "null" → NULL
#    - empty string → NULL
# ------------------------------------------------------------

string_columns = [
    "sku",
    "product_name",
    "category"
]

for column in string_columns:

    products_silver_df = products_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column).cast("string"))) == "null") |
            (F.trim(F.col(column).cast("string")) == ""),
            None
        ).otherwise(
            F.trim(F.col(column).cast("string"))
        )
    )


# ------------------------------------------------------------
# 3. Standardize SKU
#    abc-001 → ABC-001
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "sku",
    F.upper(F.col("sku"))
)


# ------------------------------------------------------------
# 4. Standardize Product Name
#    iphone 15 → Iphone 15
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_name",
    F.initcap(F.lower(F.col("product_name")))
)


# ------------------------------------------------------------
# 5. Standardize Category
#    electronics → Electronics
#    ELECTRONICS → Electronics
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "category",
    F.initcap(F.lower(F.col("category")))
)


# ------------------------------------------------------------
# 6. Cast IDs
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)


# ------------------------------------------------------------
# 7. Cast financial columns
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.col("unit_cost").cast("decimal(18,2)")
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.col("list_price").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# 8. Convert timestamp
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# 9. Handle invalid negative values
#
#    Negative cost/price is invalid for this product dataset.
#    Convert to NULL first.
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.when(
        F.col("unit_cost") < 0,
        None
    ).otherwise(
        F.col("unit_cost")
    )
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.when(
        F.col("list_price") < 0,
        None
    ).otherwise(
        F.col("list_price")
    )
)


# ------------------------------------------------------------
# 10. Business Rule
#
#     A product's cost should not be greater than
#     its selling/list price.
#
#     unit_cost > list_price → invalid
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "invalid_price",
    F.col("unit_cost") > F.col("list_price")
)

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.when(
        F.col("invalid_price"),
        None
    ).otherwise(
        F.col("unit_cost")
    )
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.when(
        F.col("invalid_price"),
        None
    ).otherwise(
        F.col("list_price")
    )
)


# ------------------------------------------------------------
# 11. Remove temporary validation column
# ------------------------------------------------------------

products_silver_df = products_silver_df.drop(
    "invalid_price"
)


# ------------------------------------------------------------
# 12. Remove rows containing NULL
# ------------------------------------------------------------

before_null_drop = products_silver_df.count()

products_silver_df = products_silver_df.dropna(
    how="any"
)

after_null_drop = products_silver_df.count()


# ------------------------------------------------------------
# 13. Preserve product history
#
# Do NOT dropDuplicates(["product_id"]).
# Product history is required by the Gold SCD Type 2 dimension.
# Only exact duplicate rows are removed.
# ------------------------------------------------------------
before_product_duplicates = products_silver_df.count()
products_silver_df = products_silver_df.dropDuplicates()
after_product_duplicates = products_silver_df.count()


# SKU uniqueness is validated, not used to delete history.
# The same product may legitimately have multiple historical versions.
before_sku_duplicates = (products_silver_df.groupBy("sku").count().filter(F.col("count") > 1).count())
after_sku_duplicates = before_sku_duplicates


# ------------------------------------------------------------
# 15. Final result
# ------------------------------------------------------------

print("=" * 70)
print("PRODUCTS DEEP CLEANING RESULT")
print("=" * 70)

print(f"Rows before NULL removal       : {before_null_drop}")
print(f"Rows after NULL removal        : {after_null_drop}")
print(f"NULL rows removed              : {before_null_drop - after_null_drop}")

print()

print(f"Rows before product duplicates : {before_product_duplicates}")
print(f"Rows after product duplicates  : {after_product_duplicates}")
print(
    f"Duplicate product_id removed   : "
    f"{before_product_duplicates - after_product_duplicates}"
)

print()

print(f"Rows before SKU duplicates     : {before_sku_duplicates}")
print(f"Rows after SKU duplicates      : {after_sku_duplicates}")
print(
    f"Duplicate SKU removed          : "
    f"{before_sku_duplicates - after_sku_duplicates}"
)

print()

print("Final Schema:")
products_silver_df.printSchema()

print("Final Data:")
products_silver_df.show(
    10,
    truncate=False
)


# ------------------------------------------------------------
# 16. Final NULL check
# ------------------------------------------------------------

print("Final NULL Check:")

products_silver_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in products_silver_df.columns
]).show()

PRODUCTS DEEP CLEANING RESULT
Rows before NULL removal       : 10000
Rows after NULL removal        : 10000
NULL rows removed              : 0

Rows before product duplicates : 10000
Rows after product duplicates  : 10000
Duplicate product_id removed   : 0

Rows before SKU duplicates     : 0
Rows after SKU duplicates      : 0
Duplicate SKU removed          : 0

Final Schema:
root
 |-- product_id: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_cost: decimal(18,2) (nullable = true)
 |-- list_price: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

Final Data:
+----------+----------+------------+-----------+---------+----------+-------------------+
|product_id|sku       |product_name|category   |unit_cost|list_price|updated_at         |
+----------+----------+------------+-----------+---------+----------+-------------------+
|85        |SKU-000085|Product

## 4.6 Stores — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [6]:
from pyspark.sql import functions as F


# ============================================================
# STORES - DEEP CLEANING
# Bronze → Silver
# ============================================================

stores_df = dfs["stores"]


stores_silver_df = stores_df.select(
    "store_id",
    "store_name",
    "city",
    "country_code",
    "opened_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["store_name", "city", "country_code"]:

    stores_silver_df = stores_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.trim(F.col(column))
        )
    )


# ------------------------------------------------------------
# Standardize store name and city
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "store_name",
    F.initcap(F.lower(F.col("store_name")))
)

stores_silver_df = stores_silver_df.withColumn(
    "city",
    F.initcap(F.lower(F.col("city")))
)


# ------------------------------------------------------------
# Standardize country
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "country_code",
    F.lower(F.trim(F.col("country_code")))
)

stores_silver_df = stores_silver_df.withColumn(
    "country_code",
    F.when(
        F.col("country_code").isin("egypt", "egy", "eg"),
        "EG"
    ).otherwise(
        F.upper(F.col("country_code"))
    )
)


# ------------------------------------------------------------
# Cast ID and date
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)

stores_silver_df = stores_silver_df.withColumn(
    "opened_at",
    F.to_date("opened_at")
)


# ------------------------------------------------------------
# Remove NULLs
# ------------------------------------------------------------

before_null = stores_silver_df.count()

stores_silver_df = stores_silver_df.dropna(
    how="any"
)

after_null = stores_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate store_id
# ------------------------------------------------------------

before_dup = stores_silver_df.count()

stores_silver_df = stores_silver_df.dropDuplicates(
    ["store_id"]
)

after_dup = stores_silver_df.count()


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("========== STORES SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

stores_silver_df.printSchema()
stores_silver_df.show(10, truncate=False)

========== STORES SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- opened_at: date (nullable = true)

+--------+---------------------+-------+------------+----------+
|store_id|store_name           |city   |country_code|opened_at |
+--------+---------------------+-------+------------+----------+
|31      |Retailpulse Store 031|City 11|EG          |2026-01-01|
|34      |Retailpulse Store 034|City 14|EG          |2026-01-01|
|28      |Retailpulse Store 028|City 08|EG          |2026-01-01|
|26      |Retailpulse Store 026|City 06|EG          |2026-01-01|
|27      |Retailpulse Store 027|City 07|EG          |2026-01-01|
|44      |Retailpulse Store 044|City 04|EG          |2026-01-01|
|12      |Retailpulse Store 012|City 12|EG          |2026-01-01|
|22      |Retailpulse Store 022|City 02|EG          

## 4.8 Order Items — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [7]:
from pyspark.sql import functions as F


# ============================================================
# ORDER_ITEMS - DEEP CLEANING
# Bronze → Silver
# ============================================================

order_items_df = dfs["order_items"]


order_items_silver_df = order_items_df.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_discount",
    "updated_at"
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

for column in ["order_item_id", "order_id"]:

    order_items_silver_df = order_items_silver_df.withColumn(
        column,
        F.col(column).cast("long")
    )


order_items_silver_df = order_items_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)


# ------------------------------------------------------------
# Cast numeric values
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "quantity",
    F.col("quantity").cast("integer")
)

order_items_silver_df = order_items_silver_df.withColumn(
    "unit_price",
    F.col("unit_price").cast("decimal(18,2)")
)

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.col("line_discount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Timestamp
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Business rules
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "quantity",
    F.when(
        F.col("quantity") <= 0,
        None
    ).otherwise(F.col("quantity"))
)

order_items_silver_df = order_items_silver_df.withColumn(
    "unit_price",
    F.when(
        F.col("unit_price") < 0,
        None
    ).otherwise(F.col("unit_price"))
)

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.when(
        F.col("line_discount") < 0,
        None
    ).otherwise(F.col("line_discount"))
)


# ------------------------------------------------------------
# Discount cannot exceed line amount
# quantity × unit_price
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.when(
        F.col("line_discount") >
        (F.col("quantity") * F.col("unit_price")),
        None
    ).otherwise(
        F.col("line_discount")
    )
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

for column in [
    "order_item_id",
    "order_id",
    "product_id"
]:

    order_items_silver_df = order_items_silver_df.withColumn(
        column,
        F.when(
            F.col(column) <= 0,
            None
        ).otherwise(F.col(column))
    )


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = order_items_silver_df.count()

order_items_silver_df = order_items_silver_df.dropna(
    how="any"
)

after_null = order_items_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate order_item_id
# ------------------------------------------------------------

before_dup = order_items_silver_df.count()

order_items_silver_df = order_items_silver_df.dropDuplicates(
    ["order_item_id"]
)

after_dup = order_items_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== ORDER_ITEMS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

order_items_silver_df.printSchema()
order_items_silver_df.show(10, truncate=False)

========== ORDER_ITEMS SILVER ==========
NULL rows removed      : 866
Duplicate rows removed : 0
root
 |-- order_item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- line_discount: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+-------------+--------+----------+--------+----------+-------------+-------------------+
|order_item_id|order_id|product_id|quantity|unit_price|line_discount|updated_at         |
+-------------+--------+----------+--------+----------+-------------+-------------------+
|26           |7       |8168      |2       |558.94    |111.79       |2026-01-01 00:28:00|
|29           |8       |9353      |4       |397.16    |0.00         |2026-01-01 00:32:00|
|474          |154     |67        |3       |485.19    |0.00         |2026-01-01 10:16:00|
|964          |320     |8434      |2       |29

## 4.10 Payments — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [8]:
from pyspark.sql import functions as F


# ============================================================
# PAYMENTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

payments_df = dfs["payments"]


payments_silver_df = payments_df.select(
    "payment_id",
    "order_id",
    "payment_method",
    "payment_status",
    "amount",
    "paid_at",
    "updated_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["payment_method", "payment_status"]:

    payments_silver_df = payments_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.lower(F.trim(F.col(column)))
        )
    )


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "payment_id",
    F.col("payment_id").cast("long")
)

payments_silver_df = payments_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)


# ------------------------------------------------------------
# Cast amount
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "amount",
    F.col("amount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "paid_at",
    F.to_timestamp("paid_at")
)

payments_silver_df = payments_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Amount cannot be negative
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "amount",
    F.when(
        F.col("amount") < 0,
        None
    ).otherwise(F.col("amount"))
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "payment_id",
    F.when(
        F.col("payment_id") <= 0,
        None
    ).otherwise(F.col("payment_id"))
)

payments_silver_df = payments_silver_df.withColumn(
    "order_id",
    F.when(
        F.col("order_id") <= 0,
        None
    ).otherwise(F.col("order_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = payments_silver_df.count()

payments_silver_df = payments_silver_df.dropna(
    how="any"
)

after_null = payments_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate payment_id
# ------------------------------------------------------------

before_dup = payments_silver_df.count()

payments_silver_df = payments_silver_df.dropDuplicates(
    ["payment_id"]
)

after_dup = payments_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== PAYMENTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

payments_silver_df.printSchema()
payments_silver_df.show(10, truncate=False)

========== PAYMENTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- paid_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+----------+--------+--------------+--------------+-------+-------------------+-------------------+
|payment_id|order_id|payment_method|payment_status|amount |paid_at            |updated_at         |
+----------+--------+--------------+--------------+-------+-------------------+-------------------+
|26        |26      |cash          |captured      |3911.78|2026-01-01 01:46:00|2026-01-01 01:44:00|
|29        |29      |card          |captured      |4638.17|2026-01-01 01:58:00|2026-01-01 01:56:00|
|474       |474     |card          |failed        |3845.55|2026-01-02 07:38:00|2026-01-02 07:36:00|


## 4.12 Fulfillment Events — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [9]:
from pyspark.sql import functions as F


# ============================================================
# FULFILLMENT_EVENTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

fulfillment_df = dfs["fulfillment_events"]


fulfillment_silver_df = fulfillment_df.select(
    "fulfillment_event_id",
    "order_id",
    "event_type",
    "event_timestamp",
    "warehouse_code",
    "updated_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["event_type", "warehouse_code"]:

    fulfillment_silver_df = fulfillment_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.lower(F.trim(F.col(column)))
        )
    )


# ------------------------------------------------------------
# IDs
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "fulfillment_event_id",
    F.col("fulfillment_event_id").cast("long")
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "fulfillment_event_id",
    F.when(
        F.col("fulfillment_event_id") <= 0,
        None
    ).otherwise(F.col("fulfillment_event_id"))
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "order_id",
    F.when(
        F.col("order_id") <= 0,
        None
    ).otherwise(F.col("order_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = fulfillment_silver_df.count()

fulfillment_silver_df = fulfillment_silver_df.dropna(
    how="any"
)

after_null = fulfillment_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate event ID
# ------------------------------------------------------------

before_dup = fulfillment_silver_df.count()

fulfillment_silver_df = fulfillment_silver_df.dropDuplicates(
    ["fulfillment_event_id"]
)

after_dup = fulfillment_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== FULFILLMENT_EVENTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

fulfillment_silver_df.printSchema()
fulfillment_silver_df.show(10, truncate=False)

========== FULFILLMENT_EVENTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- fulfillment_event_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- warehouse_code: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+--------------------+--------+----------+-------------------+--------------+-------------------+
|fulfillment_event_id|order_id|event_type|event_timestamp    |warehouse_code|updated_at         |
+--------------------+--------+----------+-------------------+--------------+-------------------+
|26                  |9       |shipped   |2026-01-01 12:36:00|wh-5          |2026-01-01 12:36:00|
|29                  |10      |shipped   |2026-01-01 12:40:00|wh-1          |2026-01-01 12:40:00|
|474                 |158     |delivered |2026-01-03 22:32:00|wh-4          |2026-01-03 22:32:00|
|964                 |322     |pa

## 4.14 Orders — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [10]:
from pyspark.sql import functions as F


# ============================================================
# ORDERS - DEEP CLEANING
# Bronze → Silver
# ============================================================

orders_df = dfs["orders"]


orders_silver_df = orders_df.select(
    "order_id",
    "customer_id",
    "store_id",
    "order_status",
    "order_timestamp",
    "order_total",
    "discount_amount",
    "updated_at"
)
# ------------------------------------------------------------
# Standardize order_status
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.lower(
        F.trim(
            F.col("order_status")
        )
    )
)

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.when(
        F.col("order_status").isin(
            "completed",
            "compeleted",
            "complete",
            "complated"
        ),
        "completed"
    )
    .when(
        F.col("order_status").isin(
            "cancelled",
            "canceled",
            "cancel"
        ),
        "cancelled"
    )
    .when(
        F.col("order_status").isin(
            "pending",
            "pendding"
        ),
        "pending"
    )
    .when(
        F.col("order_status").isin(
            "processing",
            "process"
        ),
        "processing"
    )
    .otherwise(
        F.col("order_status")
    )
)

# ------------------------------------------------------------
# Clean order_status
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.when(
        F.col("order_status").isNull() |
        (F.lower(F.trim(F.col("order_status"))) == "null") |
        (F.trim(F.col("order_status")) == ""),
        None
    ).otherwise(
        F.lower(F.trim(F.col("order_status")))
    )
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)

orders_silver_df = orders_silver_df.withColumn(
    "customer_id",
    F.col("customer_id").cast("integer")
)

orders_silver_df = orders_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)


# ------------------------------------------------------------
# Cast financial columns
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_total",
    F.col("order_total").cast("decimal(18,2)")
)

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.col("discount_amount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Convert timestamps
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_timestamp",
    F.to_timestamp("order_timestamp")
)

orders_silver_df = orders_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Negative financial values → NULL
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_total",
    F.when(
        F.col("order_total") < 0,
        None
    ).otherwise(F.col("order_total"))
)

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.when(
        F.col("discount_amount") < 0,
        None
    ).otherwise(F.col("discount_amount"))
)


# ------------------------------------------------------------
# Business rule:
# discount cannot exceed order total
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.when(
        F.col("discount_amount") > F.col("order_total"),
        None
    ).otherwise(
        F.col("discount_amount")
    )
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_id",
    F.when(F.col("order_id") <= 0, None)
    .otherwise(F.col("order_id"))
)

orders_silver_df = orders_silver_df.withColumn(
    "customer_id",
    F.when(F.col("customer_id") <= 0, None)
    .otherwise(F.col("customer_id"))
)

orders_silver_df = orders_silver_df.withColumn(
    "store_id",
    F.when(F.col("store_id") <= 0, None)
    .otherwise(F.col("store_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = orders_silver_df.count()

orders_silver_df = orders_silver_df.dropna(
    how="any"
)

after_null = orders_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate order_id
# ------------------------------------------------------------

before_dup = orders_silver_df.count()

orders_silver_df = orders_silver_df.dropDuplicates(
    ["order_id"]
)

after_dup = orders_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== ORDERS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

orders_silver_df.printSchema()
orders_silver_df.show(10, truncate=False)

========== ORDERS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- order_total: decimal(18,2) (nullable = true)
 |-- discount_amount: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+--------+-----------+--------+------------+-------------------+-----------+---------------+-------------------+
|order_id|customer_id|store_id|order_status|order_timestamp    |order_total|discount_amount|updated_at         |
+--------+-----------+--------+------------+-------------------+-----------+---------------+-------------------+
|26      |23657      |46      |cancelled   |2026-01-01 01:44:00|3911.78    |0.00           |2026-01-01 02:43:00|
|29      |26746      |21      |completed   |2026-01-01 01:56:00|4638.17    |0.00      

## 4.16 Inventory Snapshots — Bronze → Silver

Clean and standardize the source data. For customer/product entities, business-key history is intentionally preserved for SCD Type 2.

In [11]:
from pyspark.sql import functions as F


# ============================================================
# INVENTORY_SNAPSHOTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

inventory_df = dfs["inventory_snapshots"]


inventory_silver_df = inventory_df.select(
    "inventory_snapshot_id",
    "product_id",
    "store_id",
    "stock_on_hand",
    "snapshot_at",
    "updated_at"
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "inventory_snapshot_id",
    F.col("inventory_snapshot_id").cast("long")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)


# ------------------------------------------------------------
# Cast stock
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "stock_on_hand",
    F.col("stock_on_hand").cast("integer")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "snapshot_at",
    F.to_timestamp("snapshot_at")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

for column in [
    "inventory_snapshot_id",
    "product_id",
    "store_id"
]:

    inventory_silver_df = inventory_silver_df.withColumn(
        column,
        F.when(
            F.col(column) <= 0,
            None
        ).otherwise(F.col(column))
    )


# ------------------------------------------------------------
# Stock cannot be negative
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "stock_on_hand",
    F.when(
        F.col("stock_on_hand") < 0,
        None
    ).otherwise(
        F.col("stock_on_hand")
    )
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = inventory_silver_df.count()

inventory_silver_df = inventory_silver_df.dropna(
    how="any"
)

after_null = inventory_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate snapshot ID
# ------------------------------------------------------------

before_dup = inventory_silver_df.count()

inventory_silver_df = inventory_silver_df.dropDuplicates(
    ["inventory_snapshot_id"]
)

after_dup = inventory_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== INVENTORY_SNAPSHOTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

inventory_silver_df.printSchema()
inventory_silver_df.show(10, truncate=False)

========== INVENTORY_SNAPSHOTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- inventory_snapshot_id: long (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- snapshot_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+---------------------+----------+--------+-------------+-------------------+-------------------+
|inventory_snapshot_id|product_id|store_id|stock_on_hand|snapshot_at        |updated_at         |
+---------------------+----------+--------+-------------+-------------------+-------------------+
|26                   |3         |4       |109          |2026-01-06 00:00:00|2026-01-06 00:00:00|
|29                   |3         |4       |120          |2026-01-09 00:00:00|2026-01-09 00:00:00|
|474                  |48        |49      |70           |2026-01-04 00:00:00|2026-01-04 00:00:00|
|964                  |97      

## 5. Save Silver Once

Each Silver dataset is written exactly once. The `overwrite` is intentional because this notebook performs a full-refresh transformation.

In [12]:
silver_dfs = {
    "customers": customers_silver_df,
    "products": products_silver_df,
    "stores": stores_silver_df,
    "orders": orders_silver_df,
    "order_items": order_items_silver_df,
    "payments": payments_silver_df,
    "fulfillment_events": fulfillment_silver_df,
    "inventory_snapshots": inventory_silver_df
}

spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")

for table_name, df in silver_dfs.items():
    path = f"{SILVER_BASE_PATH}/{table_name}"
    (df.write.mode("overwrite").format("csv").option("header", "true").save(path))
    print(f"✓ Silver written: {table_name}")

✓ Silver written: customers
✓ Silver written: products
✓ Silver written: stores
✓ Silver written: orders
✓ Silver written: order_items
✓ Silver written: payments
✓ Silver written: fulfillment_events
✓ Silver written: inventory_snapshots


In [13]:
def register_csv_external_table(database, table, path):
    schema = spark.read.option("header", "true").option("inferSchema", "true").csv(path).schema
    columns = ",\n  ".join(
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in schema.fields
    )
    spark.sql(f"DROP TABLE IF EXISTS {database}.{table}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE {database}.{table} (
          {columns}
        )
        ROW FORMAT DELIMITED
        FIELDS TERMINATED BY ','
        STORED AS TEXTFILE
        LOCATION '{path}'
        TBLPROPERTIES ('skip.header.line.count'='1')
    """)

for table_name in silver_dfs:
    register_csv_external_table(SILVER_DB, table_name, f"{SILVER_BASE_PATH}/{table_name}")
    print(f"✓ Registered: {SILVER_DB}.{table_name}")

✓ Registered: retailpulse_silver.customers
✓ Registered: retailpulse_silver.products
✓ Registered: retailpulse_silver.stores
✓ Registered: retailpulse_silver.orders
✓ Registered: retailpulse_silver.order_items
✓ Registered: retailpulse_silver.payments
✓ Registered: retailpulse_silver.fulfillment_events
✓ Registered: retailpulse_silver.inventory_snapshots


## 6. Load Silver for Gold

Gold reads the persisted Silver layer, not the in-memory cleaning DataFrames. This makes the layer boundary explicit.

In [14]:
def read_silver(table):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{SILVER_BASE_PATH}/{table}")
    )

stores_df = read_silver("stores")
customers_df = read_silver("customers")
products_df = read_silver("products")
orders_df = read_silver("orders")
order_items_df = read_silver("order_items")
payments_df = read_silver("payments")
fulfillment_df = read_silver("fulfillment_events")
inventory_snapshots_df = read_silver("inventory_snapshots")

for name, df in {
    "customers": customers_df, "products": products_df, "stores": stores_df,
    "orders": orders_df, "order_items": order_items_df, "payments": payments_df,
    "fulfillment_events": fulfillment_df, "inventory_snapshots": inventory_snapshots_df
}.items():
    print(f"{name:25} -> {df.count():,} rows")

customers                 -> 39,415 rows
products                  -> 10,000 rows
stores                    -> 50 rows
orders                    -> 100,000 rows
order_items               -> 298,755 rows
payments                  -> 100,000 rows
fulfillment_events        -> 300,000 rows
inventory_snapshots       -> 100,000 rows


## 7. Gold — Date Dimension

In [15]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

date_range = (
    orders_df
    .select(F.to_date("order_timestamp").alias("order_date"))
    .filter(F.col("order_date").isNotNull())
    .agg(F.min("order_date").alias("min_date"), F.max("order_date").alias("max_date"))
    .select(F.date_sub("min_date", 1).alias("start_date"), F.date_add("max_date", 1).alias("end_date"))
)

dim_date = (
    date_range.select(F.explode(F.sequence(F.col("start_date"), F.col("end_date"), F.expr("interval 1 day"))).alias("full_date"))
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
)

## 8. Gold — `dim_customer` (SCD Type 2)

Every meaningful customer version is retained. `scd_valid_to` points to the next version and `is_current` identifies the latest version. The Silver layer must therefore preserve customer history.

In [16]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").asc_nulls_last())
)


customer_history = (
    customers_df
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "signup_date",
        F.to_date("signup_at")
    )
)


# Previous version
customer_history = (
    customer_history
    .withColumn(
        "previous_full_name",
        F.lag("full_name").over(customer_window)
    )
    .withColumn(
        "previous_email",
        F.lag("email").over(customer_window)
    )
    .withColumn(
        "previous_country_code",
        F.lag("country_code").over(customer_window)
    )
)


# Detect changed versions
customer_history = (
    customer_history
    .withColumn(
        "is_first_version",
        F.col("previous_full_name").isNull()
        & F.col("previous_email").isNull()
        & F.col("previous_country_code").isNull()
    )
    .withColumn(
        "has_changed",
        (
            ~F.col("full_name").eqNullSafe(
                F.col("previous_full_name")
            )
            |
            ~F.col("email").eqNullSafe(
                F.col("previous_email")
            )
            |
            ~F.col("country_code").eqNullSafe(
                F.col("previous_country_code")
            )
        )
    )
    .filter(
        F.col("is_first_version")
        | F.col("has_changed")
    )
)


# Validity
customer_scd_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").asc_nulls_last())
)


customer_history = (
    customer_history
    .withColumn(
        "scd_valid_from",
        F.col("updated_at")
    )
    .withColumn(
        "scd_valid_to",
        F.lead("updated_at").over(customer_scd_window)
    )
    .withColumn(
        "is_current",
        F.col("scd_valid_to").isNull()
    )
)


# Surrogate key
dim_customer = (
    customer_history
    .withColumn(
        "customer_key",
        F.monotonically_increasing_id()
    )
    .select(
        "customer_key",
        "customer_id",
        "full_name",
        "email",
        "country_code",
        "signup_date",
        "scd_valid_from",
        "scd_valid_to",
        "is_current"
    )
)


dim_customer.show(20, truncate=False)

+------------+-----------+-------------+------------------------+------------+-----------+-------------------+------------+----------+
|customer_key|customer_id|full_name    |email                   |country_code|signup_date|scd_valid_from     |scd_valid_to|is_current|
+------------+-----------+-------------+------------------------+------------+-----------+-------------------+------------+----------+
|0           |148        |Customer 148 |customer148@example.com |KSA         |2026-01-01 |2026-02-04 00:01:00|null        |true      |
|1           |463        |Customer 463 |customer463@example.com |UAE         |2026-01-01 |2026-02-07 16:24:00|null        |true      |
|2           |471        |Customer 471 |customer471@example.com |EG          |2026-01-01 |2026-05-21 17:50:00|null        |true      |
|3           |496        |Customer 496 |customer496@example.com |EG          |2026-01-01 |2026-05-16 03:42:00|null        |true      |
|4           |833        |Customer 833 |customer833@exa

## 9. Gold — `dim_product` (SCD Type 2)

Product versions are preserved in the same way as customers. This is required so a historical sale can resolve to the product version that was valid when the sale happened.

In [17]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("updated_at").asc_nulls_last())
)


product_history = (
    products_df
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
)


product_history = (
    product_history
    .withColumn(
        "previous_product_name",
        F.lag("product_name").over(product_window)
    )
    .withColumn(
        "previous_category",
        F.lag("category").over(product_window)
    )
    .withColumn(
        "previous_unit_cost",
        F.lag("unit_cost").over(product_window)
    )
    .withColumn(
        "previous_list_price",
        F.lag("list_price").over(product_window)
    )
)


product_history = (
    product_history
    .withColumn(
        "is_first_version",
        F.col("previous_product_name").isNull()
        & F.col("previous_category").isNull()
        & F.col("previous_unit_cost").isNull()
        & F.col("previous_list_price").isNull()
    )
    .withColumn(
        "has_changed",
        (
            ~F.col("product_name").eqNullSafe(
                F.col("previous_product_name")
            )
            |
            ~F.col("category").eqNullSafe(
                F.col("previous_category")
            )
            |
            ~F.col("unit_cost").eqNullSafe(
                F.col("previous_unit_cost")
            )
            |
            ~F.col("list_price").eqNullSafe(
                F.col("previous_list_price")
            )
        )
    )
    .filter(
        F.col("is_first_version")
        | F.col("has_changed")
    )
)


product_scd_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("updated_at").asc_nulls_last())
)


dim_product = (
    product_history
    .withColumn(
        "scd_valid_from",
        F.col("updated_at")
    )
    .withColumn(
        "scd_valid_to",
        F.lead("updated_at").over(product_scd_window)
    )
    .withColumn(
        "is_current",
        F.col("scd_valid_to").isNull()
    )
    .withColumn(
        "product_key",
        F.monotonically_increasing_id()
    )
    .select(
        "product_key",
        "product_id",
        "sku",
        "product_name",
        "category",
        "unit_cost",
        "list_price",
        "scd_valid_from",
        "scd_valid_to",
        "is_current"
    )
)


dim_product.show(20, truncate=False)

+-----------+----------+----------+------------+-----------+---------+----------+-------------------+------------+----------+
|product_key|product_id|sku       |product_name|category   |unit_cost|list_price|scd_valid_from     |scd_valid_to|is_current|
+-----------+----------+----------+------------+-----------+---------+----------+-------------------+------------+----------+
|0          |148       |SKU-000148|Product 148 |Beauty     |150.0    |182.34    |2026-06-04 04:09:00|null        |true      |
|1          |463       |SKU-000463|Product 463 |Sports     |75.67    |115.96    |2026-07-20 17:19:00|null        |true      |
|2          |471       |SKU-000471|Product 471 |Home       |346.88   |517.37    |2026-05-31 00:04:00|null        |true      |
|3          |496       |SKU-000496|Product 496 |Home       |121.81   |170.86    |2026-02-02 22:22:00|null        |true      |
|4          |833       |SKU-000833|Product 833 |Beauty     |202.23   |328.02    |2026-06-01 11:36:00|null        |true

## 10. Gold — Simple Dimensions

In [19]:
# ============================================================
# Gold — Simple Dimensions
# ============================================================

# ------------------------------------------------------------
# 1. Store Dimension
# ------------------------------------------------------------

dim_store = (
    stores_df
    .select(
        "store_id",
        "store_name",
        "city",
        "country_code",
        "opened_at"
    )
    .dropDuplicates(["store_id"])
    .withColumn("store_key", F.monotonically_increasing_id())
    .select(
        "store_key",
        "store_id",
        "store_name",
        "city",
        "country_code",
        "opened_at"
    )
)


# ------------------------------------------------------------
# 2. Latest Payment per Order
# ------------------------------------------------------------

payment_window = (
    Window
    .partitionBy(F.col("order_id"))
    .orderBy(
        F.col("paid_at").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last(),
        F.col("payment_id").desc_nulls_last()
    )
)

latest_payment = (
    payments_df
    .withColumn("paid_at", F.to_timestamp(F.col("paid_at")))
    .withColumn("updated_at", F.to_timestamp(F.col("updated_at")))
    .withColumn(
        "rn",
        F.row_number().over(payment_window)
    )
    .filter(F.col("rn") == 1)
)


# ------------------------------------------------------------
# 3. Payment Method Dimension
# ------------------------------------------------------------

dim_payment_method = (
    latest_payment
    .select(
        "payment_method",
        "payment_status"
    )
    .dropDuplicates()
    .withColumn(
        "payment_method_key",
        F.monotonically_increasing_id()
    )
    .select(
        "payment_method_key",
        "payment_method",
        "payment_status"
    )
)


# ------------------------------------------------------------
# 4. Order Status Dimension
# ------------------------------------------------------------

dim_order_status = (
    orders_df
    .select(
        F.lower(
            F.trim(
                F.col("order_status")
            )
        ).alias("order_status")
    )
    .dropDuplicates()
    .withColumn(
        "order_status_key",
        F.monotonically_increasing_id()
    )
    .select(
        "order_status_key",
        "order_status"
    )
)


# ------------------------------------------------------------
# 5. Latest Fulfillment Event per Order
# ------------------------------------------------------------

fulfillment_window = (
    Window
    .partitionBy(F.col("order_id"))
    .orderBy(
        F.col("event_timestamp").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last(),
        F.col("fulfillment_event_id").desc_nulls_last()
    )
)

latest_fulfillment = (
    fulfillment_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp(F.col("event_timestamp"))
    )
    .withColumn(
        "updated_at",
        F.to_timestamp(F.col("updated_at"))
    )
    .withColumn(
        "rn",
        F.row_number().over(fulfillment_window)
    )
    .filter(F.col("rn") == 1)
)


# ------------------------------------------------------------
# 6. Fulfillment Status Dimension
# ------------------------------------------------------------

dim_fulfillment_status = (
    latest_fulfillment
    .select(
        "event_type",
        "warehouse_code"
    )
    .dropDuplicates()
    .withColumn(
        "fulfillment_status_key",
        F.monotonically_increasing_id()
    )
    .select(
        "fulfillment_status_key",
        F.col("event_type").alias("latest_event_type"),
        "warehouse_code"
    )
)

## 11. Gold — `fact_sales`

**Grain: one row per `order_item_id`.**

Key correction: customer and product dimensions are joined using SCD validity ranges, not only `is_current = true`. This keeps historical facts attached to the correct dimension version.

In [20]:
# Base grain: one row per order item.
fact_base = (
    order_items_df.alias("oi")
    .join(orders_df.alias("o"), F.col("oi.order_id") == F.col("o.order_id"), "inner")
    .select(
        F.col("oi.order_item_id").alias("order_item_id"),
        F.col("oi.order_id").alias("order_id"),
        F.col("oi.product_id").alias("product_id"),
        F.col("oi.quantity").alias("quantity"),
        F.col("oi.unit_price").alias("unit_price"),
        F.col("oi.line_discount").alias("line_discount"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.store_id").alias("store_id"),
        F.trim(F.lower(F.col("o.order_status"))).alias("order_status"),
        F.to_timestamp("o.order_timestamp").alias("order_timestamp"),
        F.col("o.discount_amount").alias("discount_amount")
    )
    .withColumn("line_amount", F.col("quantity") * F.col("unit_price") - F.col("line_discount"))
)

order_window = Window.partitionBy("order_id")
fact_base = fact_base.withColumn("order_line_amount_total", F.sum("line_amount").over(order_window))

fact_base = fact_base.withColumn(
    "order_discount_allocated",
    F.when(
        F.col("order_line_amount_total") > 0,
        F.col("discount_amount") * F.col("line_amount") / F.col("order_line_amount_total")
    ).otherwise(F.lit(0))
)

# Payment allocation: preserve the existing project rule (all Silver payment amounts).
payment_totals = (
    payments_df.groupBy("order_id")
    .agg(F.sum("amount").alias("payment_amount_total"))
)

fact_base = fact_base.join(payment_totals, "order_id", "left")
fact_base = fact_base.withColumn(
    "payment_amount_allocated",
    F.when(
        F.col("order_line_amount_total") > 0,
        F.coalesce(F.col("payment_amount_total"), F.lit(0)) * F.col("line_amount") / F.col("order_line_amount_total")
    ).otherwise(F.lit(0))
)

# Resolve SCD2 customer version valid at the order timestamp.
customer_dim = dim_customer.alias("cd")
fact_base = (
    fact_base.alias("f")
    .join(
        customer_dim,
        (F.col("f.customer_id") == F.col("cd.customer_id"))
        & (F.col("f.order_timestamp") >= F.col("cd.scd_valid_from"))
        & (F.col("cd.scd_valid_to").isNull() | (F.col("f.order_timestamp") < F.col("cd.scd_valid_to"))),
        "left"
    )
    .select("f.*", F.col("cd.customer_key").alias("customer_key"))
)

# Resolve SCD2 product version valid at the order timestamp.
product_dim = dim_product.alias("pd")
fact_base = (
    fact_base.alias("f")
    .join(
        product_dim,
        (F.col("f.product_id") == F.col("pd.product_id"))
        & (F.col("f.order_timestamp") >= F.col("pd.scd_valid_from"))
        & (F.col("pd.scd_valid_to").isNull() | (F.col("f.order_timestamp") < F.col("pd.scd_valid_to"))),
        "left"
    )
    .select("f.*", F.col("pd.product_key").alias("product_key"), F.col("pd.unit_cost").alias("unit_cost"))
)

# Static dimensions.
fact_base = fact_base.join(dim_store.select("store_id", "store_key"), "store_id", "left")

fact_base = fact_base.join(
    latest_payment.select("order_id", "payment_method", "payment_status"),
    "order_id", "left"
)

fact_base = fact_base.join(
    latest_fulfillment.select("order_id", "event_type", "warehouse_code"),
    "order_id", "left"
)

fact_base = fact_base.join(dim_payment_method, ["payment_method", "payment_status"], "left")
fact_base = fact_base.join(dim_order_status, "order_status", "left")
fact_base = fact_base.join(
    dim_fulfillment_status,
    (fact_base.event_type == dim_fulfillment_status.latest_event_type)
    & (fact_base.warehouse_code == dim_fulfillment_status.warehouse_code),
    "left"
)

fact_sales = (
    fact_base
    .withColumn("sales_key", F.monotonically_increasing_id())
    .withColumn("date_key", F.date_format(F.to_date("order_timestamp"), "yyyyMMdd").cast("int"))
    .withColumn("margin_amount", F.col("line_amount") - F.col("unit_cost") * F.col("quantity"))
    .select(
        "sales_key", "date_key", "customer_key", "product_key", "store_key",
        "payment_method_key", "order_status_key", "fulfillment_status_key",
        "order_id", "order_item_id", "quantity", "unit_price", "line_discount",
        "line_amount", "order_discount_allocated", "payment_amount_allocated",
        "unit_cost", "margin_amount"
    )
)

## 12. Gold Validation — prove the output before publishing

These checks are intentionally executable assertions. If one fails, the notebook stops instead of publishing a misleading Gold dataset.

In [21]:
# 1) Fact grain must remain exactly one row per order_item.
source_order_items = order_items_df.select("order_item_id").count()
fact_rows = fact_sales.count()
duplicate_fact_keys = (
    fact_sales.groupBy("order_item_id").count()
    .filter(F.col("count") > 1).count()
)

assert duplicate_fact_keys == 0, f"Duplicate fact grain detected: {duplicate_fact_keys:,}"
assert fact_rows == source_order_items, (
    f"Fact row count mismatch: fact={fact_rows:,}, source order_items={source_order_items:,}"
)

# 2) SCD2 must have at most one current row per business key.
customer_current_dups = (
    dim_customer.filter(F.col("is_current") == True)
    .groupBy("customer_id").count().filter(F.col("count") > 1).count()
)
product_current_dups = (
    dim_product.filter(F.col("is_current") == True)
    .groupBy("product_id").count().filter(F.col("count") > 1).count()
)
assert customer_current_dups == 0, f"Multiple current customer versions: {customer_current_dups:,}"
assert product_current_dups == 0, f"Multiple current product versions: {product_current_dups:,}"

# 3) No overlapping SCD versions for the same business key.
customer_overlap = (
    dim_customer.alias("a")
    .join(dim_customer.alias("b"),
          (F.col("a.customer_id") == F.col("b.customer_id"))
          & (F.col("a.customer_key") != F.col("b.customer_key"))
          & F.col("a.scd_valid_to").isNotNull()
          & (F.col("a.scd_valid_from") < F.col("b.scd_valid_from"))
          & (F.col("b.scd_valid_from") < F.col("a.scd_valid_to")),
          "inner")
    .limit(1).count()
)
assert customer_overlap == 0, "Overlapping customer SCD2 intervals detected"

product_overlap = (
    dim_product.alias("a")
    .join(dim_product.alias("b"),
          (F.col("a.product_id") == F.col("b.product_id"))
          & (F.col("a.product_key") != F.col("b.product_key"))
          & F.col("a.scd_valid_to").isNotNull()
          & (F.col("a.scd_valid_from") < F.col("b.scd_valid_from"))
          & (F.col("b.scd_valid_from") < F.col("a.scd_valid_to")),
          "inner")
    .limit(1).count()
)
assert product_overlap == 0, "Overlapping product SCD2 intervals detected"

# 4) Financial allocation reconciliation.
order_discount_check = (
    fact_sales.groupBy("order_id")
    .agg(F.sum("order_discount_allocated").alias("allocated_discount"))
    .join(orders_df.select("order_id", "discount_amount"), "order_id")
    .select(F.max(F.abs(F.col("allocated_discount") - F.col("discount_amount"))).alias("max_diff"))
    .first()["max_diff"]
)

assert order_discount_check is None or float(order_discount_check) < 0.01, (
    f"Discount allocation reconciliation failed; max difference={order_discount_check}"
)

print("✓ All Gold validation checks passed")

✓ All Gold validation checks passed


## 13. Publish Gold Once

All Gold datasets use one writer function. There are no later rewrite/recreate cells. CSV headers are written once, and Hive skips the first line.

In [22]:
def publish_gold_table(df, table_name, partition_cols=None):
    path = f"{GOLD_BASE_PATH}/{table_name}"

    writer = (
        df.write.mode("overwrite")
        .format("csv")
        .option("header", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)

    non_partition_fields = [f for f in df.schema.fields if not partition_cols or f.name not in partition_cols]
    columns = ",\n  ".join(
        f"`{f.name}` {f.dataType.simpleString().upper()}"
        for f in non_partition_fields
    )

    partition_ddl = ""
    if partition_cols:
        partition_fields = [f for f in df.schema.fields if f.name in partition_cols]
        partition_ddl = "PARTITIONED BY (" + ", ".join(
            f"`{f.name}` {f.dataType.simpleString().upper()}" for f in partition_fields
        ) + ")"

    spark.sql(f"DROP TABLE IF EXISTS {GOLD_DB}.{table_name}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE {GOLD_DB}.{table_name} (
          {columns}
        )
        {partition_ddl}
        ROW FORMAT DELIMITED
        FIELDS TERMINATED BY ','
        STORED AS TEXTFILE
        LOCATION '{path}'
        TBLPROPERTIES ('skip.header.line.count'='1')
    """)

    if partition_cols:
        spark.sql(f"MSCK REPAIR TABLE {GOLD_DB}.{table_name}")

    print(f"✓ Published {GOLD_DB}.{table_name}")

# Prepare fact output and partition columns.
money_cols = ["line_amount", "order_discount_allocated", "payment_amount_allocated", "unit_cost", "margin_amount"]
fact_out = fact_sales
for c in money_cols:
    fact_out = fact_out.withColumn(c, F.col(c).cast("decimal(18,2)"))

fact_out = (
    fact_out
    .withColumn("year", (F.col("date_key") / 10000).cast("int"))
    .withColumn("month", ((F.col("date_key") % 10000) / 100).cast("int"))
)

gold_outputs = {
    "dim_date": (dim_date, None),
    "dim_customer": (dim_customer, None),
    "dim_product": (dim_product, None),
    "dim_store": (dim_store, None),
    "dim_payment_method": (dim_payment_method, None),
    "dim_order_status": (dim_order_status, None),
    "dim_fulfillment_status": (dim_fulfillment_status, None),
    "fact_sales": (fact_out, ["year", "month"]),
}

for table_name, (df, partitions) in gold_outputs.items():
    publish_gold_table(df, table_name, partitions)

✓ Published retailpulse_gold.dim_date
✓ Published retailpulse_gold.dim_customer
✓ Published retailpulse_gold.dim_product
✓ Published retailpulse_gold.dim_store
✓ Published retailpulse_gold.dim_payment_method
✓ Published retailpulse_gold.dim_order_status
✓ Published retailpulse_gold.dim_fulfillment_status
✓ Published retailpulse_gold.fact_sales


## 14. Final Gold Verification

The final checks read the published Hive tables rather than trusting the in-memory DataFrames.

In [23]:
for table_name in gold_outputs:
    count = spark.table(f"{GOLD_DB}.{table_name}").count()
    print(f"{GOLD_DB}.{table_name:25} -> {count:,} rows")

print("\nSample fact_sales:")
spark.sql(f"""
    SELECT *
    FROM {GOLD_DB}.fact_sales
    LIMIT 10
""").show(truncate=False)

print("\nSample dim_customer:")
spark.sql(f"""
    SELECT *
    FROM {GOLD_DB}.dim_customer
    ORDER BY customer_id, scd_valid_from
    LIMIT 10
""").show(truncate=False)

retailpulse_gold.dim_date                  -> 281 rows
retailpulse_gold.dim_customer              -> 39,615 rows
retailpulse_gold.dim_product               -> 10,200 rows
retailpulse_gold.dim_store                 -> 93 rows
retailpulse_gold.dim_payment_method        -> 18 rows
retailpulse_gold.dim_order_status          -> 6 rows
retailpulse_gold.dim_fulfillment_status    -> 10 rows
retailpulse_gold.fact_sales                -> 298,805 rows

Sample fact_sales:
+------------+--------+-------------+-------------+-------------+------------------+----------------+----------------------+--------+-------------+--------+----------+-------------+-----------+------------------------+------------------------+---------+-------------+----+-----+
|sales_key   |date_key|customer_key |product_key  |store_key    |payment_method_key|order_status_key|fulfillment_status_key|order_id|order_item_id|quantity|unit_price|line_discount|line_amount|order_discount_allocated|payment_amount_allocated|unit_cost|mar

## 15. Reconciliation Report

Use these numbers as the final evidence that the Gold fact preserves the source grain and financial allocation.

In [24]:
print("=== FINAL RECONCILIATION ===")
print(f"Silver order_items : {source_order_items:,}")
print(f"Gold fact_sales    : {fact_rows:,}")
print(f"Fact duplicate keys: {duplicate_fact_keys:,}")
print(f"Max discount allocation difference: {order_discount_check}")

=== FINAL RECONCILIATION ===
Silver order_items : 298,755
Gold fact_sales    : 298,755
Fact duplicate keys: 0
Max discount allocation difference: 2.2737367544323206e-13
